# Khisto Demo

This notebook demonstrates the main functionalities of the Khisto library for optimal histogram binning.

In [ ]:
import os

os.environ["KHISTO_BIN_DIR"] = (
    "/home/elouen/python/khisto-python/src/khiops/build/linux-gcc-debug/bin/khisto"
)
import numpy as np
import matplotlib.pyplot as plt

# Generate sample data
np.random.seed(42)
data = np.concatenate(
    [
        np.random.normal(-2, 0.5, 300),
        np.random.normal(1, 1.5, 700),
    ]
)
print(f"Data shape: {data.shape}")
print(f"Data range: [{data.min():.2f}, {data.max():.2f}]")

Data shape: (1000,)
Data range: [-3.62, 5.62]


## 1. NumPy-like API: `khisto.histogram`

Drop-in replacement for `numpy.histogram` with optimal binning.

In [2]:
from khisto import histogram

# Compute optimal histogram
hist, bin_edges = histogram(data)

print(f"Number of bins: {len(hist)}")
print(f"Bin edges: {bin_edges}")
print(f"Frequencies: {hist}")

FileNotFoundError: [Errno 2] No such file or directory: 'khisto'

In [ ]:
# Compare with numpy fixed-width bins
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# NumPy histogram (fixed bins)
np_hist, np_edges = np.histogram(data, bins=20)
axes[0].stairs(np_hist, np_edges, fill=True, alpha=0.7)
axes[0].set_title("NumPy (20 fixed bins)")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Count")

# Khisto histogram (optimal bins)
axes[1].stairs(hist, bin_edges, fill=True, alpha=0.7, color="orange")
axes[1].set_title(f"Khisto ({len(hist)} optimal bins)")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# With density normalization
density, bin_edges = histogram(data, density=True)

# Verify normalization (integral should be ~1)
widths = np.diff(bin_edges)
integral = np.sum(density * widths)
print(f"Integral of density: {integral:.6f}")

In [ ]:
# With max_bins limit
hist_limited, edges_limited = histogram(data, max_bins=5)
print(f"Limited to max 5 bins: got {len(hist_limited)} bins")
print(f"Bin edges: {edges_limited}")

In [ ]:
# With range specification
hist_range, edges_range = histogram(data, range=(-3, 3))
print(f"Range [-3, 3]: {len(hist_range)} bins")
print(f"Bin edges: {edges_range}")

## 2. Matplotlib API: `khisto.matplotlib.hist`

Plot optimal histograms with a matplotlib-like interface.

In [ ]:
from khisto.matplotlib import hist

# Basic histogram plot
fig, ax = plt.subplots(figsize=(8, 5))
n, bins, patches = hist(data, ax=ax)
ax.set_xlabel("Value")
ax.set_ylabel("Count")
ax.set_title("Optimal Histogram")
plt.show()

print(f"Returned {len(n)} bin values")

In [ ]:
# Density plot
fig, ax = plt.subplots(figsize=(8, 5))
n, bins, patches = hist(data, density=True, ax=ax, color="green")
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.set_title("Optimal Histogram (Density)")
plt.show()

In [ ]:
# Different histogram types
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

hist(data, histtype="bar", ax=axes[0], color="steelblue")
axes[0].set_title('histtype="bar"')

hist(data, histtype="step", ax=axes[1], color="red")
axes[1].set_title('histtype="step"')

hist(data, histtype="stepfilled", ax=axes[2], color="purple", alpha=0.5)
axes[2].set_title('histtype="stepfilled"')

plt.tight_layout()
plt.show()

In [ ]:
# Horizontal orientation
fig, ax = plt.subplots(figsize=(6, 6))
hist(data, orientation="horizontal", ax=ax, color="coral")
ax.set_xlabel("Count")
ax.set_ylabel("Value")
ax.set_title("Horizontal Histogram")
plt.show()

## 3. Core API: `compute_histogram` and `HistogramResult`

Direct access to detailed histogram information.

In [ ]:
from khisto.core import compute_histogram

# Compute histogram with full details
result = compute_histogram(data)

print(f"Type: {type(result)}")
print(f"Number of bins: {len(result.frequency)}")
print(f"Is best (optimal): {result.is_best}")
print(f"Granularity: {result.granularity}")

In [ ]:
# Access all histogram attributes
print("=== Bin Boundaries ===")
print(f"Lower bounds: {result.lower_bound}")
print(f"Upper bounds: {result.upper_bound}")

print("\n=== Computed Properties ===")
print(f"Bin edges: {result.bin_edges}")
print(f"Bin widths: {result.bin_widths}")
print(f"Bin centers: {result.bin_centers}")

print("\n=== Values ===")
print(f"Frequency: {result.frequency}")
print(f"Probability: {result.probability}")
print(f"Density: {result.density}")

In [ ]:
# Visualize using HistogramResult
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Frequency
axes[0].bar(result.bin_centers, result.frequency, width=result.bin_widths * 0.9)
axes[0].set_title("Frequency")
axes[0].set_xlabel("Value")

# Probability
axes[1].bar(
    result.bin_centers, result.probability, width=result.bin_widths * 0.9, color="green"
)
axes[1].set_title("Probability")
axes[1].set_xlabel("Value")

# Density
axes[2].bar(
    result.bin_centers, result.density, width=result.bin_widths * 0.9, color="orange"
)
axes[2].set_title("Density")
axes[2].set_xlabel("Value")

plt.tight_layout()
plt.show()

## 4. Multi-Scale Histograms: `return_all=True`

Get all granularity levels computed by the algorithm.

In [ ]:
# Get all granularity levels
results = compute_histogram(data, return_all=True)

print(f"Number of granularity levels: {len(results)}")
print("\nGranularity levels:")
for r in results:
    marker = " ← BEST" if r.is_best else ""
    print(f"  Granularity {r.granularity}: {len(r.frequency)} bins{marker}")

In [ ]:
# Visualize different granularities
n_levels = min(6, len(results))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, r in enumerate(results[:n_levels]):
    ax = axes[i]
    ax.stairs(r.frequency, r.bin_edges, fill=True, alpha=0.7)
    title = f"Granularity {r.granularity} ({len(r.frequency)} bins)"
    if r.is_best:
        title += " ★ BEST"
        ax.set_facecolor("#ffffee")
    ax.set_title(title)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## Summary

Khisto provides three API levels:

1. **`khisto.histogram`** - NumPy-compatible, returns `(hist, bin_edges)`
2. **`khisto.matplotlib.hist`** - Matplotlib-compatible, plots directly
3. **`khisto.core.compute_histogram`** - Full access to `HistogramResult` with all details